<a href="https://colab.research.google.com/github/ReneeKang/pandas_pyspark_pycaret/blob/main/Spark%20Shuffle.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
import pyspark.sql.types as T

spark = (
    SparkSession.builder
    .appName("JoinStrategyDemo")
    .getOrCreate()
)

In [4]:
import random
from datetime import datetime, timedelta

# 샘플 데이터 생성 함수
def generate_sample_data(num_records):
    """
    샘플 데이터를 생성합니다.

    - product_id
    - category: (A, B, C)
    - amount: 판매 금액
    """
    data = []
    categories = ['A', 'B', 'C']

    for i in range(num_records):
        # 낮은 카디널리티 컬럼
        category = random.choice(categories)

        # 높은 카디널리티 컬럼 (거의 모든 값이 고유)
        product_id = f"PROD_{i:06d}"

        # 판매 금액
        amount = round(random.uniform(10.0, 1000.0), 2)

        # 타임스탬프 (높은 카디널리티)
        timestamp = datetime.now() - timedelta(days=random.randint(0, 365))

        data.append((category, product_id, amount, timestamp.strftime("%Y-%m-%d %H:%M:%S")))

    return data

# 데이터 생성
sample_data = generate_sample_data(1000000)

# DataFrame 생성
schema = T.StructType([
    T.StructField("category", T.StringType(), True),
    T.StructField("product_id", T.StringType(), True),
    T.StructField("amount", T.DoubleType(), True),
    T.StructField("timestamp", T.StringType(), True)
])

df = spark.createDataFrame(sample_data, schema)

print(f"생성된 레코드 수: {df.count()}")
df.show(10, truncate=False)

# 데이터 분포 확인
print("\n카테고리별 레코드 수:")
df.groupBy("category").count().show()

생성된 레코드 수: 1000000
+--------+-----------+------+-------------------+
|category|product_id |amount|timestamp          |
+--------+-----------+------+-------------------+
|A       |PROD_000000|267.17|2025-05-30 05:42:03|
|C       |PROD_000001|651.09|2025-06-07 05:42:03|
|A       |PROD_000002|108.25|2026-01-27 05:42:03|
|C       |PROD_000003|468.04|2026-03-28 05:42:03|
|B       |PROD_000004|310.23|2025-08-01 05:42:03|
|C       |PROD_000005|321.16|2026-03-28 05:42:03|
|B       |PROD_000006|873.19|2025-12-05 05:42:03|
|C       |PROD_000007|887.18|2025-05-22 05:42:03|
|B       |PROD_000008|352.4 |2025-06-28 05:42:03|
|B       |PROD_000009|669.26|2025-06-29 05:42:03|
+--------+-----------+------+-------------------+
only showing top 10 rows

카테고리별 레코드 수:
+--------+------+
|category| count|
+--------+------+
|       B|333097|
|       C|333217|
|       A|333686|
+--------+------+



## Narrow Transformation

#### 👉 한 파티션의 데이터가 다른 파티션으로 이동하지 않는 변환

| 메소드                 | 설명              |
| ------------------- | --------------- |
| `select`            | 컬럼 선택           |
| `selectExpr`        | SQL 표현식으로 컬럼 선택 |
| `withColumn`        | 컬럼 추가 또는 수정     |
| `withColumnRenamed` | 컬럼 이름 변경        |
| `drop`              | 컬럼 제거           |
| `filter` / `where`  | 조건 필터링          |
| `na.fill`           | 결측값 채우기         |
| `na.drop`           | 결측값 제거          |
| `limit`             | 상위 N개 row 반환    |
| `sample`            | 샘플링             |
| `map` (RDD)         | row 단위 변환       |
| `flatMap` (RDD)     | row 펼치기         |
| `mapPartitions`     | 파티션 단위 처리       |
| `foreachPartition`  | 파티션 단위 액션       |


### 1) filter
- 각 파티션에서 독립적으로 필터링하므로 Shuffle이 발생하지 않습니다.

In [5]:
filtered_df = df.filter(F.col("category") == "A")

# 실행 계획 확인
print("=== Filter 실행 계획 ===")
filtered_df.explain()

=== Filter 실행 계획 ===
== Physical Plan ==
*(1) Filter (isnotnull(category#2) AND (category#2 = A))
+- *(1) Scan ExistingRDD[category#2,product_id#3,amount#4,timestamp#5]




### 2) select
- 필요한 컬럼만 선택하므로 Shuffle이 발생하지 않습니다.

In [6]:
selected_df = df.select("category", "amount")

print("=== Select 실행 계획 ===")
selected_df.explain()

=== Select 실행 계획 ===
== Physical Plan ==
*(1) Project [category#2, amount#4]
+- *(1) Scan ExistingRDD[category#2,product_id#3,amount#4,timestamp#5]




### 3) withColumn
- 새 컬럼을 추가하므로 Shuffle이 발생하지 않습니다

In [7]:
df_with_new_col = df.withColumn(
    "amount_category",
    F.when(F.col("amount") > 500, "High")
    .when(F.col("amount") > 100, "Medium")
    .otherwise("Low")
)

print("=== WithColumn 실행 계획 ===")
df_with_new_col.explain()

=== WithColumn 실행 계획 ===
== Physical Plan ==
*(1) Project [category#2, product_id#3, amount#4, timestamp#5, CASE WHEN (amount#4 > 500.0) THEN High WHEN (amount#4 > 100.0) THEN Medium ELSE Low END AS amount_category#42]
+- *(1) Scan ExistingRDD[category#2,product_id#3,amount#4,timestamp#5]




## Wide Transformation

#### 👉 같은 키의 데이터를 모으기 위해 파티션 간 데이터 이동이 필요한 변환

| 메소드                      | 설명         |
| ------------------------ | ---------- |
| `groupBy`                | 키 기준 집계    |
| `agg`                    | 집계 함수 적용   |
| `count` (groupBy 이후)     | 그룹별 개수     |
| `join`                   | 키 기준 조인    |
| `leftJoin`               | 왼쪽 기준 조인   |
| `rightJoin`              | 오른쪽 기준 조인  |
| `fullOuterJoin`          | 전체 조인      |
| `crossJoin`              | 모든 조합 조인   |
| `distinct`               | 전체 중복 제거   |
| `dropDuplicates`         | 중복 row 제거  |
| `orderBy`                | 전체 정렬      |
| `sort`                   | 전체 정렬      |
| `repartition`            | 파티션 재분배    |
| `coalesce(shuffle=True)` | 강제 파티션 재분배 |


### 1) groupBy
- 같은 키를 가진 데이터를 같은 파티션으로 모으기 위해 Shuffle이 발생합니다.

In [8]:
grouped_df = df.groupBy("category").agg(
    F.count("*").alias("count"),
    F.sum("amount").alias("total_amount")
)

# 실행 계획 확인
print("=== GroupBy 실행 계획 ===")
grouped_df.explain()

=== GroupBy 실행 계획 ===
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[category#2], functions=[count(1), sum(amount#4)])
   +- Exchange hashpartitioning(category#2, 200), ENSURE_REQUIREMENTS, [plan_id=129]
      +- HashAggregate(keys=[category#2], functions=[partial_count(1), partial_sum(amount#4)])
         +- Project [category#2, amount#4]
            +- Scan ExistingRDD[category#2,product_id#3,amount#4,timestamp#5]




### 2) join
- 두 데이터셋을 키로 결합하기 위해 Shuffle이 발생합니다.

In [9]:
# 작은 테이블 생성 (조인용)
category_info = spark.createDataFrame([
    ("A", "Electronics"),
    ("B", "Clothing"),
    ("C", "Food")
], ["category", "category_name"])

# 조인 수행
joined_df = df.join(category_info, on="category", how="inner")

print("=== Join 실행 계획 ===")
joined_df.explain()

=== Join 실행 계획 ===
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [category#2, product_id#3, amount#4, timestamp#5, category_name#56]
   +- SortMergeJoin [category#2], [category#55], Inner
      :- Sort [category#2 ASC NULLS FIRST], false, 0
      :  +- Exchange hashpartitioning(category#2, 200), ENSURE_REQUIREMENTS, [plan_id=156]
      :     +- Filter isnotnull(category#2)
      :        +- Scan ExistingRDD[category#2,product_id#3,amount#4,timestamp#5]
      +- Sort [category#55 ASC NULLS FIRST], false, 0
         +- Exchange hashpartitioning(category#55, 200), ENSURE_REQUIREMENTS, [plan_id=157]
            +- Filter isnotnull(category#55)
               +- Scan ExistingRDD[category#55,category_name#56]




### 3) orderBy
- 전체 데이터를 정렬하기 위해 Shuffle이 발생합니다.

In [10]:
sorted_df = df.orderBy("amount")

print("=== OrderBy 실행 계획 ===")
sorted_df.explain()

=== OrderBy 실행 계획 ===
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Sort [amount#4 ASC NULLS FIRST], true, 0
   +- Exchange rangepartitioning(amount#4 ASC NULLS FIRST, 200), ENSURE_REQUIREMENTS, [plan_id=170]
      +- Scan ExistingRDD[category#2,product_id#3,amount#4,timestamp#5]




## Shuffle이 성능에 미치는 영향

#### 👉 불필요하게 많은 Shuffle은 연산 속도를 느리게 한다.

In [12]:
# 예제 1: 낮은 카디널리티 컬럼으로 그룹화
# category 컬럼은 A, B, C 세 가지만 있으므로 낮은 카디널리티입니다
from time import time


low_cardinality_result = df.groupBy("category").agg(
    F.count("*").alias("count")
)

start = time()
low_cardinality_result.count()
print(f"실행 시간: {time() - start}")

실행 시간: 3.3690643310546875


In [13]:
# 예제 2: 높은 카디널리티 컬럼으로 그룹화
high_cardinality_result = df.groupBy("product_id").agg(
    F.count("*").alias("count")
)

start = time()
high_cardinality_result.count()
print(f"실행 시간: {time() - start}")

실행 시간: 7.679209470748901
